# Unidade III — Pré-processamento de Dados

## Transformação, codificação e discretização

**Carga estimada:** 3 horas  
**Pré-requisitos:** pandas, medidas de posição e fluxo supervisionado básico.

> **Pergunta norteadora:** como representar atributos em uma forma adequada ao algoritmo sem usar informação que estaria indisponível no momento da previsão?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- diferenciar normalização min–max e padronização;
- transformar distribuições assimétricas e codificar categorias;
- comparar discretização por largura e por frequência;
- usar pipelines ajustados somente com o conjunto de treino.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Escala e forma

A normalização min–max leva um valor $x$ ao intervalo desejado:

$$x' = \frac{x-x_{\min}}{x_{\max}-x_{\min}}.$$

A padronização usa média $\mu$ e desvio-padrão $\sigma$:

$$z = \frac{x-\mu}{\sigma}.$$

Min–max é sensível aos extremos e pode produzir valores fora do intervalo em novos dados. Padronização não torna a distribuição normal; apenas muda centro e escala. Para valores não negativos e assimétricos, `log1p` comprime a cauda e aceita zero.


### Por que usar distribuições diferentes na simulação?

O código a seguir cria uma base **sintética** de 500 clientes. Cada gerador foi escolhido para reproduzir uma característica plausível do tipo de atributo, e não porque toda base real necessariamente segue essas distribuições:

- `rng.lognormal(mean=8.2, sigma=0.7, size=n)` gera a `renda`. Uma variável lognormal assume apenas valores positivos e costuma apresentar muitos valores moderados e poucos valores muito altos, formando uma cauda à direita. Os parâmetros `mean` e `sigma` pertencem à distribuição normal do **logaritmo** da renda; portanto, `8.2` e `0.7` não são a média e o desvio-padrão da renda na escala monetária.
- `rng.poisson(1.8, n)` gera o número de `chamados`. A distribuição de Poisson é apropriada para simular **contagens** de ocorrências em um intervalo, produzindo inteiros não negativos. O parâmetro $\lambda=1{,}8$ representa a quantidade média esperada de chamados por cliente nesta simulação. Em uma Poisson ideal, média e variância são iguais a $\lambda$; dados reais podem não satisfazer essa hipótese.
- `rng.binomial(1, probabilidade)` gera `cancelou`. Como o primeiro argumento é 1, realizamos um único ensaio de Bernoulli para cada cliente: o resultado é `1` com a probabilidade calculada e `0` caso contrário. Assim, clientes com a mesma probabilidade ainda podem ter resultados diferentes. Isso representa um desfecho probabilístico, não uma regra determinística.

Antes da geração binomial, `logito` combina chamados e renda em um escore. A expressão `1 / (1 + np.exp(-logito))` aplica a função logística e converte esse escore em uma probabilidade entre 0 e 1. Essa fórmula foi definida por nós para **simular** o alvo; nenhum modelo foi treinado.

Os outros geradores cumprem papéis mais simples: `rng.integers` produz idades inteiras no intervalo especificado, enquanto `rng.choice` sorteia cidades segundo as proporções informadas. Depois, algumas idades são substituídas por `NaN` para permitir a demonstração de imputação.


In [2]:
n = 500
dados = pd.DataFrame({
    "idade": rng.integers(18, 76, n).astype(float),
    "renda": rng.lognormal(mean=8.2, sigma=0.7, size=n),
    "cidade": rng.choice(["Recife", "Olinda", "Paulista"], n, p=[0.55, 0.25, 0.20]),
    "chamados": rng.poisson(1.8, n),
})
dados.loc[rng.choice(n, 24, replace=False), "idade"] = np.nan
logito = -2.0 + 0.45 * dados["chamados"] - 0.00012 * dados["renda"]
dados["cancelou"] = rng.binomial(1, 1 / (1 + np.exp(-logito)))
dados.head()


,idade,renda,cidade,chamados,cancelou
0,23.0,3408.461216,Olinda,1,0
1,62.0,1063.786643,Olinda,1,0
2,55.0,1303.839078,Paulista,1,0
3,43.0,16162.887183,Recife,0,0
4,43.0,1478.530833,Paulista,3,0


### Para que servem `log1p` e `skew`?

A renda simulada possui cauda à direita: poucos valores altos ficam muito afastados da maioria. A transformação

$$x' = \log(1+x)$$

é calculada por `np.log1p(x)`. Somar 1 permite transformar o valor zero, pois $\log(1+0)=0$, e a função `log1p` também oferece boa precisão numérica para valores muito próximos de zero. Para dados não negativos, ela preserva a ordem e **comprime diferenças entre valores altos**. Isso pode reduzir a influência visual e numérica de uma cauda longa, mas não remove observações, não garante normalidade e muda a escala em que os resultados devem ser interpretados.

O método `skew` calcula a **assimetria amostral**. Uma assimetria positiva indica cauda mais longa à direita; negativa, cauda mais longa à esquerda; valor próximo de zero indica distribuição aproximadamente simétrica quanto a essa medida. Não existe um limite universal que, sozinho, determine se a assimetria é aceitável. Neste exemplo, vamos comparar `skew` antes e depois de `log1p` para medir se a transformação reduziu a assimetria da renda.

> **Atenção:** média, mediana e desvio-padrão das duas colunas estão em escalas diferentes. Eles ajudam a descrever cada representação, mas não devem ser comparados como se tivessem a mesma unidade.


In [3]:
resumo_renda = pd.DataFrame({
    "original": dados["renda"],
    "log1p": np.log1p(dados["renda"]),
}).agg(["mean", "median", "std", "skew"]).round(2)
resumo_renda


,original,log1p
mean,4533.22,8.17
median,3601.40,8.19
std,3535.01,0.71
skew,2.26,-0.06


Na saída, a renda original apresenta `skew` positivo e elevado, coerente com a cauda à direita criada pela lognormal. Depois de `log1p`, o valor fica próximo de zero, indicando que a representação transformada é muito mais simétrica nesta amostra. Esse resultado era esperado pela forma como os dados foram simulados, mas deve sempre ser verificado em vez de presumido para um conjunto real.


## Categorias e discretização

Codificar cidades como 0, 1 e 2 introduziria uma ordem artificial. A codificação *one-hot* cria um indicador por categoria. Categorias novas exigem uma política explícita; abaixo, `handle_unknown="ignore"` produz zeros nos indicadores conhecidos.

Discretizar converte um atributo numérico em intervalos. Larguras iguais facilitam interpretar a escala, mas podem gerar intervalos vazios; frequências aproximadamente iguais equilibram contagens, mas os limites dependem da amostra e valores iguais podem dificultar a divisão. Há perda de informação dentro de cada intervalo.


In [4]:
renda = dados[["renda"]]
por_largura = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
por_frequencia = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
comparacao_bins = pd.DataFrame({
    "largura_igual": por_largura.fit_transform(renda).ravel().astype(int),
    "frequencia_igual": por_frequencia.fit_transform(renda).ravel().astype(int),
})
comparacao_bins.apply(pd.Series.value_counts).fillna(0).astype(int)


,largura_igual,frequencia_igual
0,426,125
1,61,125
2,10,125
3,3,125


## Pipeline e prevenção de vazamento

Vazamento ocorre quando o treinamento recebe informação que não estaria legitimamente disponível na aplicação. Calcular mediana, média, desvio ou categorias antes da partição deixa o teste influenciar o preparo. A sequência correta é: separar primeiro; ajustar (`fit`) transformações no treino; apenas aplicar (`transform`) ao teste. Um `Pipeline` registra essa ordem e deve envolver também o modelo em uma avaliação futura.


In [5]:
X = dados.drop(columns="cancelou")
y = dados["cancelou"]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numericas = ["idade", "renda", "chamados"]
categoricas = ["cidade"]
pipeline_numerico = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler()),
])
preprocessador = ColumnTransformer([
    ("num", pipeline_numerico, numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
])
X_treino_pronto = preprocessador.fit_transform(X_treino)
X_teste_pronto = preprocessador.transform(X_teste)
pd.Series({"linhas_treino": len(X_treino_pronto), "linhas_teste": len(X_teste_pronto), "atributos_saida": X_treino_pronto.shape[1]})


linhas_treino      375
linhas_teste       125
atributos_saida      6
dtype: int64

A mediana armazenada pelo imputador, as estatísticas do escalonador e as categorias do codificador vieram exclusivamente das 375 linhas de treino. A variável-alvo não foi usada pelas transformações. Discretização supervisionada, diferentemente, pode usar o alvo para escolher cortes, mas precisa ficar dentro do mesmo fluxo de treino e validação.

> **U03-NB02-V01 — Verifique seu entendimento:** por que chamar `fit_transform` separadamente no treino e no teste é incorreto, mesmo sem usar explicitamente a variável-alvo?

> **U03-NB02-E01 — Exercício:** recupere do pipeline a mediana aprendida para `idade` e compare-a à mediana do teste. Explique qual delas deve ser usada para transformar o teste e por quê.


## Síntese

- Escalonamento altera representação, não corrige qualidade nem garante normalidade.
- Codificação deve respeitar o significado do atributo.
- Discretização simplifica, mas perde resolução e aprende limites da amostra.
- Todo parâmetro de pré-processamento deve ser aprendido sem acesso ao teste.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seções 2.5–2.6.
- SCIKIT-LEARN DEVELOPERS. *User Guide*: preprocessing data e pipelines. Versão 1.x.
